# NewsSum V2: extractive baselines vs fine-tuned transformers vs a small LLM on BBC News Summary

This notebook runs every experiment of [NewsSum V2](https://github.com/MHKhan50/NewsSum-Bench):

1. **Data audit**: near-duplicate articles, leakage under a naive random split, and how the reference summaries were built (extractive? lead-biased?).
2. **Baselines**: Lead-3, length-matched Lead, length-matched TextRank, and an extractive oracle (upper bound).
3. **Zero-shot models**: BART-large-CNN, PEGASUS-CNN/DailyMail and Qwen2.5-1.5B-Instruct.
4. **Fine-tuned models** (3 seeds each): BART-base, BART-large, T5-base, Flan-T5-base, Flan-T5-large, PEGASUS-large, and Qwen2.5-1.5B-Instruct with LoRA.
5. **Cross-category test**: BART-base trained on four categories and tested on the fifth.
6. **Evaluation**: ROUGE-1/2/L/Lsum with bootstrap CIs, paired comparison with the Lead baseline, BERTScore, copying (coverage, density, novel bigrams) and numbers not supported by the article.

### How to run on Kaggle
- **Settings → Accelerator: GPU T4 x2** (two jobs run in parallel, one per GPU), **Internet: On**.
- **Add Input → Datasets**: `pariza/bbc-news-summary`.
- First time: set `MODE = "smoke"` and run all (about 10 minutes) to check the setup. Then set `MODE = "full"` and **Save Version → Save & Run All (Commit)**.
- **Next session (resume):** *Add Input → Notebooks →* this notebook's previous version (its output). Finished jobs are skipped and unfinished ones continue from their checkpoint. Repeat until the status table shows everything as `done`.

In [ ]:
# Install the project package (no dependency changes to Kaggle's PyTorch) and the metric/LoRA libraries
!pip install -q --no-deps git+https://github.com/MHKhan50/NewsSum-Bench.git
!pip install -q rouge-score bert-score peft

In [ ]:
import time
T0 = time.time()

from pathlib import Path
import shutil
import pandas as pd

from newssum.pipeline import prepare_data, run_baselines, add_bertscore
from newssum.runner import plan, launch, status, load_prepared
from newssum import report

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

MODE = "full"          # "smoke": quick check on a few articles; "full": all experiments
HOURS = 11.3           # wall-clock budget for this session (Kaggle stops GPU sessions at 12 h)
SEEDS = 3              # seeds per fine-tuned model
REPORT_RESERVE = 0.4   # hours kept for BERTScore, tables and saving the output

SMOKE = MODE == "smoke"
OUT = Path("/kaggle/working/newssum_smoke" if SMOKE else "/kaggle/working/newssum")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Environment used for this run (reported in the paper)
import importlib.metadata as md, platform, torch
for pkg in ["newssum", "torch", "transformers", "peft", "rouge-score", "bert-score", "pandas", "numpy"]:
    try:
        print(f"{pkg:13s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:13s} not installed")
print("python       ", platform.python_version())
print("GPUs         ", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "none")

## Resume from a previous session

If an earlier version of this notebook was added as input, its finished runs, checkpoints and saved models are copied back so nothing is trained twice.

In [ ]:
prev = sorted({p.parents[2] for p in Path("/kaggle/input").rglob("runs/*/result.json")
               if p.parents[2].name == OUT.name}, key=lambda p: len(str(p)))
if prev and not (OUT / "runs").exists():
    print("Resuming from", prev[0])
    shutil.copytree(prev[0], OUT, dirs_exist_ok=True)
else:
    print("Starting fresh" if not prev else "Output folder already present")

## 1. Data and audit

Articles are loaded from the Kaggle dataset; the headline is removed from the model input (it is the first line of each file). Near-duplicate articles (Jaccard similarity of word 5-grams at least 0.9) are removed *before* splitting, and the split (70/15/15, stratified by category, seed 42) is fixed for all models.

In [ ]:
if not (OUT / "data" / "meta.json").exists():
    audit_tables = prepare_data(OUT)
df, meta = load_prepared(OUT)
print(meta)
display(pd.read_csv(OUT / "results/audit_overview.csv"))
display(pd.read_csv(OUT / "results/audit_by_category.csv"))
display(pd.read_csv(OUT / "results/audit_leakage_random_split.csv"))
display(pd.read_csv(OUT / "results/audit_selection_by_position.csv").T)

## 2. Extractive baselines and oracle (CPU)

The length-matched baselines get a word budget of `ratio x article length`, where `ratio` is the median summary/article length ratio on the **training** split. The oracle picks article sentences greedily to maximise ROUGE against the reference, so it is an upper bound for any extractive method.

In [ ]:
if not (OUT / "runs" / "oracle__main__s0" / "result.json").exists():
    display(run_baselines(OUT, smoke=SMOKE))

## 3. Zero-shot and fine-tuned models (GPU)

One worker process per GPU takes jobs from the list below, cheapest first. Fine-tuning uses fixed hyperparameters (see `src/newssum/models.py`) with early stopping on validation loss; the test set is used once, for the final summaries. When the time budget runs out, a running job saves a checkpoint and continues in the next session.

In [ ]:
jobs = plan("smoke" if SMOKE else "full", seeds=SEEDS)
st = status(OUT, jobs)
display(st.groupby(["model", "status"]).size().unstack(fill_value=0))

In [ ]:
hours_left = HOURS - (time.time() - T0) / 3600 - REPORT_RESERVE
print(f"GPU budget for this session: {hours_left:.2f} h")
st = launch(OUT, jobs, hours=hours_left, smoke=SMOKE)
display(st.groupby(["model", "status"]).size().unstack(fill_value=0))

## 4. BERTScore and result tables

In [ ]:
n = add_bertscore(OUT, batch_size=16)
print(f"BERTScore added to {n} runs")
tables = report.write_all(OUT)
cols = ["system", "type", "seeds", "rouge1", "rouge2", "rougeL", "rougeLsum", "rougeLsum_sd",
        "rougeLsum_lo", "rougeLsum_hi", "bertscore", "words", "coverage", "density", "novel_bigrams",
        "unsupported_numbers_pct", "vs_lead-ratio_rougeLsum", "vs_lead-ratio_rougeLsum_lo", "vs_lead-ratio_rougeLsum_hi"]
main = tables["main"]
display(main[[c for c in cols if c in main.columns]])

In [ ]:
display(tables["by_category"])
display(tables["cross_category"])
display(tables["training"])

In [ ]:
from IPython.display import Image
for f in ["rouge_lsum_by_system.png", "copying_vs_rouge2.png", "reference_position.png"]:
    p = OUT / "results/figures" / f
    if p.exists():
        display(Image(filename=str(p), width=650))

## 5. Example summaries

In [ ]:
doc = df[df["split"] == "test"].iloc[0]
print(doc["doc_id"], "|", doc["title"], "\n\nREFERENCE:\n", doc["summary"][:800])
for run in ["lead-ratio__main__s0", "textrank-ratio__main__s0", "bart-large-cnn__main__s0",
            "bart-base__main__s0", "qwen2.5-1.5b-lora__main__s0"]:
    p = OUT / "runs" / run / "predictions.jsonl"
    if p.exists():
        pr = pd.read_json(p, lines=True, dtype={"doc_id": str}).set_index("doc_id")
        print(f"\n{run.split('__')[0].upper()}:\n", pr.loc[doc['doc_id'], "prediction"][:800])

## 6. Status and results bundle

`newssum_results.zip` holds the tables, figures, per-document scores and predictions (no model weights). It is the file to copy into the GitHub repository's `results/` folder.

In [ ]:
st = status(OUT, jobs)
pending = st[st["status"] != "done"]
print(f"{len(st) - len(pending)}/{len(st)} jobs done")
if len(pending):
    display(pending)
    print("Not finished: add this version's output as input and run the notebook again.")

import zipfile
bundle = OUT.parent / ("newssum_smoke_results.zip" if SMOKE else "newssum_results.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        rel = p.relative_to(OUT)
        if p.is_file() and rel.parts[0] in ("results", "runs", "data") and "ckpt" not in rel.parts \
                and p.name != "dataset.jsonl":
            z.write(p, rel)
print(bundle, f"{bundle.stat().st_size / 1e6:.1f} MB")
print(f"total notebook time {(time.time() - T0) / 3600:.2f} h")